# Weekly Laser and CBP Checkout (BLOCK-T683)

This notebook evaluates the outcome of the weekly calibration system test **BLOCK-T683**, which exercises the Tunable Laser and the Collimated Beam Projector (CBP).

The block performs the following steps:
1. **Laser functional test** – runs the `laser_functional` sequence, scanning wavelengths and recording electrometer signals
2. **CBP motion test** – exercises azimuth/elevation moves, focus changes, mask changes, and mask rotation changes
3. **Laser + CBP combined test** – runs the `laser_cbp` sequence with both systems active

**Expected CBP motion sequence:**
- Azimuth: unpark → 15.0° → -15.0° → park
- Elevation: unpark → 0.0° → -20.0° → park
- Focus: 5000 → 3800
- Mask: 5 → 1
- Mask rotation: 275.0° → 96.5°

In [ ]:
# User input
day_obs = 20250410

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
import base64
import io
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy.io import fits
from astropy.time import Time, TimeDelta
from IPython.display import HTML, display
from lsst.resources import ResourcePath
from lsst_efd_client import EfdClient

In [ ]:
date_str = str(day_obs)
date_str_fmt = f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:]}T23:59:00.00"
end_time = Time(date_str_fmt, format="isot")
start_time = end_time - TimeDelta(60.0 * 60 * 24, format="sec")

client = EfdClient("usdf_efd")
print(f"Querying EFD from {start_time.iso} to {end_time.iso}")

In [ ]:
def collapsable_figure(fig, title, open_by_default=False):
    """Save fig as base64 PNG inside a collapsable HTML <details> element, then close it."""
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", dpi=100)
    plt.close(fig)
    buf.seek(0)
    img_b64 = base64.b64encode(buf.read()).decode("utf-8")
    open_attr = "open" if open_by_default else ""
    display(HTML(
        f'<details {open_attr} style="margin:6px 0">'
        f'<summary style="cursor:pointer;font-weight:bold;padding:4px 0">{title}</summary>'
        f'<img src="data:image/png;base64,{img_b64}" style="max-width:100%;margin-top:8px"/>'
        f'</details>'
    ))

TABLE_STYLES = [
    {"selector": "th", "props": [
        ("background-color", "#1f4e79"), ("color", "white"),
        ("font-weight", "bold"), ("text-align", "center"), ("padding", "8px 12px"),
    ]},
    {"selector": "td", "props": [("padding", "6px 14px"), ("text-align", "center")]},
    {"selector": "tr:nth-child(even)", "props": [("background-color", "#f5f5f5")]},
    {"selector": "caption", "props": [
        ("font-size", "14px"), ("font-weight", "bold"),
        ("padding-bottom", "8px"), ("text-align", "left"),
    ]},
]

## 1. Electrometer Data

Electrometer signals from the `laser_functional` and `laser_cbp` sequences.
Images are identified by the block ID **BT683** in the electrometer large-file event IDs.
The earlier exposures (by timestamp) correspond to `laser_functional`; the later ones to `laser_cbp`.

In [ ]:
msg_log_topic = "lsst.sal.Electrometer.logevent_largeFileObjectAvailable"
elec_df = await client.select_time_series(
    msg_log_topic, ["url", "id", "salIndex"], start=start_time, end=end_time
)

if len(elec_df) > 0:
    block_df = elec_df[elec_df["id"].str.contains("BT683", case=False, na=False)].copy()
    block_df["dayobs"] = block_df["id"].str.split("_").str[2]
    day_block_df = block_df[block_df["dayobs"] == str(day_obs)].sort_index()
    print(f"Found {len(day_block_df)} electrometer exposures for BLOCK-T683 on {day_obs}")
else:
    day_block_df = pd.DataFrame()
    print(f"No electrometer data found on {day_obs}")

day_block_df

In [ ]:
# Mean electrometer current per file vs time, all BT683 files per instrument,
# color-coded by laser wavelength (300–1100 nm, cycling if more files than steps).
wavelengths = list(np.arange(300, 1200, 100))  # [300, 400, ..., 1100]
cmap = plt.cm.rainbow
norm = plt.Normalize(300, 1100)

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

for ax, sal_idx in zip(axes, [102, 103]):
    sub_df = day_block_df[day_block_df["salIndex"] == sal_idx].sort_index()
    for i, (ts, row) in enumerate(sub_df.iterrows()):
        wl = wavelengths[i % len(wavelengths)]
        try:
            path = ResourcePath("s3://lfa@" + row.url.split(".org/")[1])
            with path.open("rb") as f:
                hdu = fits.open(f)
                mean_signal = np.mean(hdu[1].data["Signal"])
            ax.scatter([ts], [mean_signal], color=cmap(norm(wl)), s=100, zorder=5)
        except Exception as e:
            print(f"Electrometer {sal_idx} file {i}: {e}")
    ax.set_ylabel("Mean Current (A)")
    ax.set_title(f"Electrometer {sal_idx} ({len(sub_df)} exposures)")

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
axes[-1].set_xlabel("Time (UTC)")
fig.suptitle(f"Electrometer Mean Current vs Time – DayObs: {day_obs}", fontsize=13)

fig.subplots_adjust(right=0.84, hspace=0.35)
cax = fig.add_axes([0.86, 0.15, 0.025, 0.7])
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, cax=cax)
cbar.set_label("Laser Wavelength (nm)")
cbar.set_ticks(wavelengths)

collapsable_figure(fig, "Electrometer Mean Current vs Time")

## 2. FiberSpectrograph Spectra (laser_functional)

Spectra recorded during the `laser_functional` sequence from both Blue and Red spectrographs. The laser steps through **8 wavelengths from 300 to 1000 nm** (center 700 nm ± 400 nm, step 100 nm). Each trace is labeled with its commanded laser wavelength.

In [ ]:
# FiberSpectrograph files are identified by timestamp only, not block ID.
# Use the full span of all BT683 electrometer exposures to bracket the search window.
if len(day_block_df) > 0:
    block_sorted = day_block_df.sort_index()
    t_fs_start = Time(block_sorted.index[0].to_pydatetime()) - TimeDelta(120, format="sec")
    t_fs_end = Time(block_sorted.index[-1].to_pydatetime()) + TimeDelta(300, format="sec")

    fs_df = await client.select_time_series(
        "lsst.sal.FiberSpectrograph.logevent_largeFileObjectAvailable",
        ["url", "salIndex"],
        start=t_fs_start,
        end=t_fs_end,
    )
    day_fs_df = fs_df.sort_index() if len(fs_df) > 0 else pd.DataFrame()
    print(f"Found {len(day_fs_df)} FiberSpectrograph exposures")
    print(f"  Window: {t_fs_start.iso} → {t_fs_end.iso}")
else:
    day_fs_df = pd.DataFrame()
    print("No electrometer data available to bracket the FiberSpectrograph time window")

day_fs_df

In [ ]:
# Plot all FiberSpectrograph spectra grouped by instrument.
# Wavelength labels cycle through 300–1100 nm in 100 nm steps.
laser_wavelengths = list(np.arange(300, 1200, 100))  # [300, 400, ..., 1100]

if len(day_fs_df) > 0:
    sal_indices = sorted(day_fs_df["salIndex"].unique())
    fig, axes = plt.subplots(1, len(sal_indices), figsize=(7 * len(sal_indices), 5), squeeze=False)
    axes = axes[0]

    for ax, sal_idx in zip(axes, sal_indices):
        sub_df = day_fs_df[day_fs_df["salIndex"] == sal_idx].sort_index()
        for i, (ts, row) in enumerate(sub_df.iterrows()):
            laser_wl = laser_wavelengths[i % len(laser_wavelengths)]
            try:
                path = ResourcePath("s3://lfa@" + row.url.split(".org/")[1])
                with path.open("rb") as f:
                    hdu = fits.open(f)
                    if len(hdu) > 1 and hdu[1].columns is not None:
                        wl_col = next((c for c in hdu[1].columns.names if "wave" in c.lower()), None)
                        fl_col = next(
                            (c for c in hdu[1].columns.names if c.lower() in ("flux", "spectrum", "signal")),
                            None,
                        )
                        if wl_col and fl_col:
                            wavelength = hdu[1].data[wl_col]
                            flux = hdu[1].data[fl_col]
                        else:
                            raise ValueError(f"Unexpected columns: {hdu[1].columns.names}")
                    else:
                        flux = hdu[0].data.ravel()
                        crval1 = hdu[0].header.get("CRVAL1", 0)
                        cdelt1 = hdu[0].header.get("CDELT1", 1)
                        crpix1 = hdu[0].header.get("CRPIX1", 1)
                        wavelength = crval1 + cdelt1 * (np.arange(len(flux)) - crpix1 + 1)
                ax.plot(wavelength, flux, label=f"{laser_wl} nm")
            except Exception as e:
                print(f"FiberSpectrograph {sal_idx} step {i} ({laser_wl} nm): {e}")

        ax.set_xlabel("Wavelength (nm)")
        ax.set_ylabel("Flux")
        ax.set_title(f"FiberSpectrograph {sal_idx} ({len(sub_df)} exposures)")
        ax.legend(fontsize=8, ncol=2)

    fig.suptitle(
        f"BLOCK-T683 FiberSpectrograph Spectra – DayObs: {day_obs}", fontsize=12
    )
    plt.tight_layout()
    collapsable_figure(fig, f"FiberSpectrograph Spectra – DayObs: {day_obs}")
else:
    print("No FiberSpectrograph data to plot.")

## 3. TunableLaser Wavelength

The TunableLaser steps through multiple wavelengths during the `laser_functional` and `laser_cbp` sequences.

In [ ]:
try:
    laser_df = await client.select_time_series(
        "lsst.sal.TunableLaser.wavelength", ["wavelength"], start=start_time, end=end_time
    )
    if len(laser_df) > 0:
        fig, ax = plt.subplots(figsize=(13, 4))
        ax.step(laser_df.index, laser_df["wavelength"], where="post", linewidth=1.5)
        ax.set_xlabel("Time (UTC)")
        ax.set_ylabel("Wavelength (nm)")
        ax.set_title(f"TunableLaser Wavelength – DayObs: {day_obs}")
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
        plt.xticks(rotation=45)
        plt.tight_layout()
        collapsable_figure(fig, f"TunableLaser Wavelength – DayObs: {day_obs}")
        print(f"Wavelengths visited: {sorted(laser_df['wavelength'].unique())}")
    else:
        print(f"No TunableLaser wavelength data for {day_obs}")
except Exception as e:
    print(f"Error querying TunableLaser wavelength: {e}")

## 4. Laser Temperature and Humidity

Temperature sensors (ESS salIndex 107) inside the laser enclosure, plus relative humidity. High humidity or temperature excursions during the test can affect laser stability.

In [ ]:
temp_fields = ["salIndex", "location"] + [f"temperatureItem{i}" for i in range(8)]
temp_df = await client.select_time_series(
    "lsst.sal.ESS.temperature", temp_fields, start=start_time, end=end_time
)
temp_df = temp_df[temp_df["salIndex"] == 404]

hum_df = await client.select_time_series(
    "lsst.sal.ESS.relativeHumidity", ["salIndex", "relativeHumidityItem"],
    start=start_time, end=end_time,
)
hum_df = hum_df[hum_df["salIndex"] == 404]

if len(temp_df) > 0:
    # Rows with location "Laser Enclosure" are metadata-only; drop them
    temp_df = temp_df[temp_df["location"] != "Laser Enclosure"].copy()
    locations = temp_df.iloc[0]["location"].split()
    temp_df = temp_df.rename(columns={f"temperatureItem{i}": loc for i, loc in enumerate(locations)})
    plot_locs = [loc for loc in locations if loc != "LaserFan"]

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

    ax1.plot(temp_df.index, temp_df[plot_locs])
    ax1.set_ylabel("Temperature (°C)")
    ax1.set_title(f"Laser Enclosure Temperature – DayObs: {day_obs}")
    ax1.legend(plot_locs, bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)

    if len(hum_df) > 0:
        ax2.plot(hum_df.index, hum_df["relativeHumidityItem"], color="C2")
    else:
        ax2.text(0.5, 0.5, "No humidity data", transform=ax2.transAxes, ha="center")
    ax2.set_ylabel("Relative Humidity (%)")
    ax2.set_title("Laser Enclosure Relative Humidity")

    ax2.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax2.set_xlabel("Time (UTC)")
    plt.setp(ax1.get_xticklabels(), visible=False)
    plt.setp(ax2.get_xticklabels(), rotation=45)
    plt.tight_layout()
    collapsable_figure(fig, f"Laser Temperature and Humidity – DayObs: {day_obs}")
else:
    print(f"No laser temperature data (ESS salIndex 404) for {day_obs}")

## 5. CBP Motion Test

Verify that the CBP moved to each commanded position during the block. 
Dashed lines indicate the expected commanded values.

In [ ]:
# CBP azimuth and elevation – actual telemetry vs. commanded positions + error vs time
expected_az = [15.0, -15.0]
expected_el = [0.0, -20.0]

cbp_az_df = await client.select_time_series(
    "lsst.sal.CBP.azimuth", ["azimuth"], start=start_time, end=end_time
)
cbp_el_df = await client.select_time_series(
    "lsst.sal.CBP.elevation", ["elevation"], start=start_time, end=end_time
)
cbp_cmd_df = await client.select_time_series(
    "lsst.sal.CBP.command_move", ["azimuth", "elevation"], start=start_time, end=end_time
)
cbp_cmd_df = cbp_cmd_df.sort_index()

fig = plt.figure(figsize=(13, 11))
gs = fig.add_gridspec(3, 1, height_ratios=[3, 3, 2], hspace=0.45)
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1], sharex=ax1)
ax3 = fig.add_subplot(gs[2], sharex=ax1)

# ── Azimuth ──────────────────────────────────────────────────────────────────
if len(cbp_az_df) > 0:
    ax1.plot(cbp_az_df.index, cbp_az_df["azimuth"], linewidth=1.5, label="Actual")
if len(cbp_cmd_df) > 0:
    ax1.scatter(cbp_cmd_df.index, cbp_cmd_df["azimuth"], marker="*", s=200,
                color="C3", zorder=5, label="Commanded")
for val in expected_az:
    ax1.axhline(val, linestyle="--", alpha=0.5, label=f"Expected: {val}°")
ax1.set_ylabel("Azimuth (°)")
ax1.set_title(f"CBP Azimuth – DayObs: {day_obs}")
ax1.legend(fontsize=8)
plt.setp(ax1.get_xticklabels(), visible=False)

# ── Elevation ─────────────────────────────────────────────────────────────────
if len(cbp_el_df) > 0:
    ax2.plot(cbp_el_df.index, cbp_el_df["elevation"], color="C1", linewidth=1.5, label="Actual")
if len(cbp_cmd_df) > 0:
    ax2.scatter(cbp_cmd_df.index, cbp_cmd_df["elevation"], marker="*", s=200,
                color="C3", zorder=5, label="Commanded")
for val in expected_el:
    ax2.axhline(val, linestyle="--", alpha=0.5, label=f"Expected: {val}°")
ax2.set_ylabel("Elevation (°)")
ax2.set_title(f"CBP Elevation – DayObs: {day_obs}")
ax2.legend(fontsize=8)
plt.setp(ax2.get_xticklabels(), visible=False)

# ── Error vs time (twinx: az left, el right) ──────────────────────────────────
ax3_r = ax3.twinx()

if len(cbp_cmd_df) > 0 and len(cbp_az_df) > 0 and len(cbp_el_df) > 0:
    # Forward-fill the commanded step function onto each telemetry timestamp
    merged_az = pd.merge_asof(
        cbp_az_df[["azimuth"]].sort_index(),
        cbp_cmd_df[["azimuth"]].rename(columns={"azimuth": "cmd"}).sort_index(),
        left_index=True, right_index=True, direction="backward",
    ).dropna()
    merged_el = pd.merge_asof(
        cbp_el_df[["elevation"]].sort_index(),
        cbp_cmd_df[["elevation"]].rename(columns={"elevation": "cmd"}).sort_index(),
        left_index=True, right_index=True, direction="backward",
    ).dropna()

    az_err = merged_az["azimuth"] - merged_az["cmd"]
    el_err = merged_el["elevation"] - merged_el["cmd"]

    ax3.plot(az_err.index, az_err, color="C0", linewidth=1.5, label="Azimuth error")
    ax3.axhline(0, color="black", linewidth=0.8, alpha=0.4)
    ax3.set_ylabel("Azimuth Error (°)", color="C0")
    ax3.tick_params(axis="y", labelcolor="C0")

    ax3_r.plot(el_err.index, el_err, color="C1", linewidth=1.5, label="Elevation error")
    ax3_r.set_ylabel("Elevation Error (°)", color="C1")
    ax3_r.tick_params(axis="y", labelcolor="C1")

    for t_cmd in cbp_cmd_df.index:
        ax3.axvline(t_cmd, color="C3", linestyle="--", alpha=0.4, linewidth=1)

    lines_l, labels_l = ax3.get_legend_handles_labels()
    lines_r, labels_r = ax3_r.get_legend_handles_labels()
    ax3.legend(lines_l + lines_r, labels_l + labels_r, fontsize=8)
else:
    ax3.text(0.5, 0.5, "Insufficient data to compute errors",
             transform=ax3.transAxes, ha="center")

ax3.set_title("Position Error vs Time (Actual − Commanded step function)")
ax3.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
ax3.set_xlabel("Time (UTC)")
plt.setp(ax3.get_xticklabels(), rotation=45)

collapsable_figure(fig, f"CBP Azimuth and Elevation – DayObs: {day_obs}")

In [ ]:
# CBP focus telemetry
# Expected: 5000 then 3800
expected_focus = [5000, 3800]

cbp_focus_df = await client.select_time_series(
    "lsst.sal.CBP.focus", ["focus"], start=start_time, end=end_time
)

fig, ax = plt.subplots(figsize=(13, 4))
if len(cbp_focus_df) > 0:
    ax.plot(cbp_focus_df.index, cbp_focus_df["focus"], linewidth=1.5, label="Actual")
    for val in expected_focus:
        ax.axhline(val, linestyle="--", alpha=0.5, label=f"Expected: {val}")
    ax.legend()
else:
    ax.text(0.5, 0.5, f"No CBP focus data for {day_obs}", transform=ax.transAxes, ha="center")
ax.set_title(f"CBP Focus – DayObs: {day_obs}")
ax.set_ylabel("Focus")
ax.set_xlabel("Time (UTC)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
plt.xticks(rotation=45)
plt.tight_layout()
collapsable_figure(fig, f"CBP Focus – DayObs: {day_obs}")

In [ ]:
# CBP mask and mask rotation – actual telemetry (string names) vs. commanded values (integers)
MASK_MAP = {1: "1mm-pinhole", 2: "2-lines", 3: "empty", 4: "one-per-CCD", 5: "one-per-amp"}

expected_masks = [5, 1]        # from BLOCK-T683
expected_rotations = [275.0, 96.5]

cbp_mask_df = await client.select_time_series(
    "lsst.sal.CBP.mask", ["mask", "rotation"], start=start_time, end=end_time
)
cbp_cmd_mask_df = await client.select_time_series(
    "lsst.sal.CBP.command_changeMask", ["mask"], start=start_time, end=end_time
)
cbp_cmd_rot_df = await client.select_time_series(
    "lsst.sal.CBP.command_changeMaskRotation", ["rotation"], start=start_time, end=end_time
)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

# Mask – map string names → integers so the step plot is numeric; label y-ticks with names
name_to_num = {v: k for k, v in MASK_MAP.items()}
if len(cbp_mask_df) > 0:
    mask_numeric = cbp_mask_df["mask"].map(name_to_num)
    ax1.step(cbp_mask_df.index, mask_numeric, where="post", linewidth=2, label="Actual")
if len(cbp_cmd_mask_df) > 0:
    ax1.scatter(cbp_cmd_mask_df.index, pd.to_numeric(cbp_cmd_mask_df["mask"], errors="coerce"),
                marker="*", s=200, color="C3", zorder=5, label="Commanded")
for val in expected_masks:
    ax1.axhline(val, linestyle="--", alpha=0.5, label=f"Expected: {MASK_MAP[val]}")
ax1.set_yticks(list(MASK_MAP.keys()))
ax1.set_yticklabels(list(MASK_MAP.values()))
ax1.set_ylabel("Mask")
ax1.set_title(f"CBP Mask – DayObs: {day_obs}")
ax1.legend()

# Rotation
if len(cbp_mask_df) > 0:
    ax2.step(cbp_mask_df.index, cbp_mask_df["rotation"], where="post",
             linewidth=2, color="C2", label="Actual")
if len(cbp_cmd_rot_df) > 0:
    ax2.scatter(cbp_cmd_rot_df.index, cbp_cmd_rot_df["rotation"], marker="*", s=200,
                color="C3", zorder=5, label="Commanded")
for val in expected_rotations:
    ax2.axhline(val, linestyle="--", alpha=0.5, label=f"Expected: {val}°")
ax2.set_ylabel("Mask Rotation (°)")
ax2.set_title(f"CBP Mask Rotation – DayObs: {day_obs}")
ax2.legend()

ax2.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
ax2.set_xlabel("Time (UTC)")
plt.xticks(rotation=45)
plt.tight_layout()
collapsable_figure(fig, f"CBP Mask and Rotation – DayObs: {day_obs}")

### CBP Temperature

Four temperature sensors inside the CBP enclosure (ESS salIndex 405). Sensor names are not yet assigned.

In [ ]:
cbp_temp_df = await client.select_time_series(
    "lsst.sal.ESS.temperature",
    ["salIndex"] + [f"temperatureItem{i}" for i in range(4)],
    start=start_time,
    end=end_time,
)
cbp_temp_df = cbp_temp_df[cbp_temp_df["salIndex"] == 405]

if len(cbp_temp_df) > 0:
    fig, ax = plt.subplots(figsize=(13, 5))
    for i in range(4):
        col = f"temperatureItem{i}"
        if col in cbp_temp_df.columns:
            ax.plot(cbp_temp_df.index, cbp_temp_df[col], label=f"Sensor {i}")
    ax.set_ylabel("Temperature (°C)")
    ax.set_xlabel("Time (UTC)")
    ax.set_title(f"CBP Temperature (ESS salIndex 405) – DayObs: {day_obs}")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    plt.xticks(rotation=45)
    ax.legend()
    plt.tight_layout()
    collapsable_figure(fig, f"CBP Temperature – DayObs: {day_obs}")
else:
    print(f"No CBP temperature data (ESS salIndex 405) for {day_obs}")

### CBP Final State

Summary of the last commanded vs last observed value for each CBP axis. The **Δ** column is color-coded: green (|Δ| < 0.001), yellow (< 1.0), red (≥ 1.0). Focus expected final value (3800) is taken from the block definition; all others use the last `command_*` event.

In [ ]:
# CBP final state: last commanded vs last actual for each axis
MASK_MAP = {1: "1mm-pinhole", 2: "2-lines", 3: "empty", 4: "one-per-CCD", 5: "one-per-amp"}

def safe_last(df, col):
    return df[col].iloc[-1] if len(df) > 0 and col in df.columns else None

def num_delta(a, b):
    try:
        return float(a) - float(b)
    except (TypeError, ValueError):
        return None

def fmt(v, dec=3):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return "—"
    if isinstance(v, str):
        return v
    return f"{v:.{dec}f}"

act_az  = safe_last(cbp_az_df,       "azimuth")
cmd_az  = safe_last(cbp_cmd_df,      "azimuth")
act_el  = safe_last(cbp_el_df,       "elevation")
cmd_el  = safe_last(cbp_cmd_df,      "elevation")
act_foc = safe_last(cbp_focus_df,    "focus")
cmd_foc = 3800.0  # last setFocus in BLOCK-T683
act_msk = safe_last(cbp_mask_df,     "mask")   # string e.g. "1mm-pinhole"
cmd_msk = safe_last(cbp_cmd_mask_df, "mask")   # integer e.g. 1
act_rot = safe_last(cbp_mask_df,     "rotation")
cmd_rot = safe_last(cbp_cmd_rot_df,  "rotation")

cmd_msk_name = MASK_MAP.get(int(cmd_msk), str(cmd_msk)) if cmd_msk is not None else None
mask_delta = "match" if act_msk == cmd_msk_name else "mismatch"

cbp_status = pd.DataFrame([
    {"Parameter": "Azimuth (°)",       "Last Commanded": fmt(cmd_az, 3),      "Last Actual": fmt(act_az, 4),  "Δ": fmt(num_delta(act_az,  cmd_az),  4)},
    {"Parameter": "Elevation (°)",     "Last Commanded": fmt(cmd_el, 3),      "Last Actual": fmt(act_el, 4),  "Δ": fmt(num_delta(act_el,  cmd_el),  4)},
    {"Parameter": "Focus",             "Last Commanded": fmt(cmd_foc, 0),     "Last Actual": fmt(act_foc, 1), "Δ": fmt(num_delta(act_foc, cmd_foc), 1)},
    {"Parameter": "Mask",              "Last Commanded": fmt(cmd_msk_name),   "Last Actual": fmt(act_msk),    "Δ": mask_delta},
    {"Parameter": "Mask Rotation (°)", "Last Commanded": fmt(cmd_rot, 2),     "Last Actual": fmt(act_rot, 2), "Δ": fmt(num_delta(act_rot, cmd_rot), 4)},
]).set_index("Parameter")

def color_delta(v):
    if v == "match":
        return "background-color: #c8e6c9; color: #1b5e20"
    if v == "mismatch":
        return "background-color: #ffcdd2; color: #b71c1c; font-weight: bold"
    try:
        fv = float(v)
        if abs(fv) < 0.001:
            return "background-color: #c8e6c9; color: #1b5e20"
        elif abs(fv) < 1.0:
            return "background-color: #fff9c4; color: #e65100"
        else:
            return "background-color: #ffcdd2; color: #b71c1c"
    except (TypeError, ValueError):
        return ""

display(
    cbp_status.style
    .applymap(color_delta, subset=["Δ"])
    .set_table_styles(TABLE_STYLES)
    .set_caption(f"CBP Final State – DayObs: {day_obs}")
)

## 6. Component Summary States

Verify that all calibration system components successfully transitioned:
**STANDBY → ENABLED** at the start and **ENABLED → STANDBY** at the end of the block.

SAL summary state values: `1`=DISABLED, `2`=ENABLED, `3`=FAULT, `4`=OFFLINE, `5`=STANDBY

In [ ]:
state_names = {1: "DISABLED", 2: "ENABLED", 3: "FAULT", 4: "OFFLINE", 5: "STANDBY"}

components = [
    ("LEDProjector", None),
    ("TunableLaser", None),
    ("FiberSpectrograph", 101),
    ("FiberSpectrograph", 102),
    ("Electrometer", 103),
    ("Electrometer", 102),
    ("LinearStage", 101),
    ("LinearStage", 102),
    ("LinearStage", 103),
    ("LinearStage", 104),
    ("CBP", None),
]

rows = []
for csc, sal_index in components:
    topic = f"lsst.sal.{csc}.logevent_summaryState"
    label = f"{csc}:{sal_index}" if sal_index else csc
    try:
        df = await client.select_time_series(
            topic, ["summaryState", "salIndex"], start=start_time, end=end_time
        )
        if sal_index is not None and "salIndex" in df.columns:
            df = df[df["salIndex"] == sal_index]
        if len(df) > 0:
            first_state = df["summaryState"].iloc[0]
            last_state = df["summaryState"].iloc[-1]
            states_seen = df["summaryState"].map(lambda x: state_names.get(x, str(x))).tolist()
            rows.append({
                "Component": label,
                "First State": state_names.get(first_state, first_state),
                "Last State": state_names.get(last_state, last_state),
                "All States": " → ".join(states_seen),
                "FAULT?": "YES" if 3 in df["summaryState"].values else "no",
            })
        else:
            rows.append({"Component": label, "First State": "no data", "Last State": "no data",
                         "All States": "", "FAULT?": "n/a"})
    except Exception as e:
        rows.append({"Component": label, "First State": f"error: {e}", "Last State": "",
                     "All States": "", "FAULT?": "n/a"})

state_table = pd.DataFrame(rows).set_index("Component")

STATE_COLORS = {
    "ENABLED":  "background-color: #c8e6c9; color: #1b5e20; font-weight: bold",
    "STANDBY":  "background-color: #e3f2fd; color: #1565c0",
    "FAULT":    "background-color: #ffcdd2; color: #b71c1c; font-weight: bold",
    "DISABLED": "background-color: #fff9c4; color: #e65100",
    "OFFLINE":  "background-color: #f3e5f5; color: #6a1b9a",
    "YES":      "background-color: #ffcdd2; color: #b71c1c; font-weight: bold",
    "no":       "background-color: #e8f5e9; color: #2e7d32",
    "no data":  "background-color: #eeeeee; color: #757575; font-style: italic",
}

def color_state_cell(v):
    return STATE_COLORS.get(str(v), "")

display(
    state_table.style
    .applymap(color_state_cell)
    .set_table_styles(TABLE_STYLES)
    .set_caption(f"Component Summary States – DayObs: {day_obs}")
)